# Day 14 — Simplified: How GPT Trains (Plain English Version)

## The Question

How do LLMs like GPT learn to write? **What's their training task?**

## The Answer (1 sentence)

> "Given some text, predict what word comes next."

That's it. That's the whole training task. GPT, Claude, Llama — all trained on this.

## A Bigram is the Simplest Version

A **bigram model** says: "To predict the next word, I'll only look at the IMMEDIATELY PREVIOUS one."

```
After "the" → most likely next word is... ?
After "I"   → most likely next word is... ?
After "to"  → most likely next word is... ?
```

It's obviously stupid (no context!), but it shows you the framework. Real LLMs do the same task, just with more context.

## How To "Train" a Bigram (Without a Neural Net!)

```
1. Walk through your text
2. For every consecutive pair (a, b), increment count[a][b]
3. Normalize each row → probability distribution

Done. No gradient descent needed. Just counting.
```

After training:
```
count["the"]["cat"] = 47
count["the"]["dog"] = 31
count["the"]["sun"] = 12
...

P("cat" | "the") = 47 / (sum of row) = ~15%
```

## How to "Generate" Text

```
1. Start with a word: "the"
2. Look up P(next | "the") → sample a word, say "cat"
3. Now current word = "cat"
4. Look up P(next | "cat") → sample → "sat"
5. Repeat
```

You just generated: "the cat sat ..."

This same loop is what GPT does. Just with more context.

## The Neural Bigram (For Practice)

We CAN do bigrams without a neural network. But we'll build a neural version because:
- The same architecture scales to better models
- It uses Cross-Entropy loss (same as GPT)
- It uses softmax sampling (same as GPT)

The "neural bigram" is literally `nn.Embedding(vocab, vocab)`:
- Row `i` of the embedding = the logits for "what comes after word i"
- After training, softmax those logits = probability distribution

```python
class BigramLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)
    def forward(self, x):
        return self.embedding(x)   # logits for next word
```

That tiny class IS a language model.

## Key Vocabulary

- **Cross-Entropy loss** — `-log(P[correct_next_word])`. Penalizes the model when the right word has low probability.
- **Perplexity** — `exp(loss)`. "How many tokens does the model effectively choose between?" Lower = better.
- **Temperature** — divide logits by T before softmax. Low T = predictable, high T = creative.

These three concepts appear in every LLM paper.

## Why Bigrams Fail

```
"the boy sat on the m_"   ← bigram only sees "m"
"the dog ran on the m_"   ← bigram only sees "m"

These should predict DIFFERENT things ("man" vs "mat"?), but the
bigram gives the SAME distribution. It has no context.
```

The next 6 days (Days 15-20) all build toward fixing this — letting the model use the WHOLE context, not just 1 token.

## TL;DR

```
Bigram = "predict next word from 1 previous word"
GPT    = "predict next word from THOUSANDS of previous words"

Same task. Same loss. Same sampling. Just way more context.
```

See `notebook.ipynb` for the full version with counting + neural + generation + perplexity.